# GRU trên Colab — phạm vi đã chốt

Notebook này chạy **đúng bốn tổ hợp** đã chốt ngày 23/09/2026, trước khi xem bất kỳ kết quả nào:

| Môi trường | Tầm dự báo | Chế độ | Hạt giống |
|---|---|---|---|
| E1 | 1 | N1 | 42 |
| E1 | 12 | N1 | 42 |
| E2 | 1 | N1 | 42 |
| E2 | 12 | N1 | 42 |

Mô hình: một lớp GRU 32 đơn vị ẩn, cửa sổ đầu vào 288 bước, hàm mất mát tuyệt đối, tối đa 30 epoch, dừng sớm với patience 5.
Chuẩn hoá N1 là trung bình và độ lệch chuẩn **phần train của từng máy**, giống hệt các mô hình bảng; dự đoán đưa về CPU% bằng `ŷ = ẑ·sd + mu`.

## Cần tải lên

Một tệp duy nhất: `goi_colab.zip`, sinh ở máy bằng

```bash
python scripts/xuat_goi_colab.py
```

## Sẽ tải về

`ket_qua_gru.zip`, đặt vào `results/colab/` rồi chạy

```bash
python scripts/nhap_ket_qua_colab.py --zip results/colab/ket_qua_gru.zip
```

## Chống mất việc khi Colab ngắt

Kết quả ghi thẳng vào Google Drive sau **từng tổ hợp**. Colab ngắt giữa chừng thì chạy lại ô cuối với `--resume`, phần đã xong được bỏ qua.

## 1. Kiểm GPU

Không có GPU thì **dừng tại đây**: GRU với cửa sổ 288 bước chạy bằng CPU mất khoảng 20 phút mỗi epoch, tức hàng chục giờ cho bốn tổ hợp.
Vào `Runtime > Change runtime type > Hardware accelerator > GPU` rồi chạy lại ô này.

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout[:800])
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Không có GPU. Đổi runtime sang GPU rồi chạy lại.")
print("GPU:", torch.cuda.get_device_name(0))

## 2. Thư viện

Colab có sẵn torch, pandas, numpy và pyarrow. Ô này chỉ in phiên bản để ghi vào báo cáo, và cài `pyarrow` nếu thiếu.

In [ ]:
try:
    import pyarrow
except ImportError:
    !pip -q install pyarrow
    import pyarrow
import torch, pandas, numpy
print("torch", torch.__version__)
print("pandas", pandas.__version__, "| numpy", numpy.__version__, "| pyarrow", pyarrow.__version__)

## 3. Gắn Drive và tải gói dữ liệu lên

Kết quả ghi vào Drive nên không mất khi Colab ngắt.

In [ ]:
from google.colab import drive, files
from pathlib import Path
drive.mount('/content/drive')
LAM_VIEC = Path('/content/drive/MyDrive/cwp_gru')
LAM_VIEC.mkdir(parents=True, exist_ok=True)
print('thư mục làm việc:', LAM_VIEC)

goi = LAM_VIEC / 'goi_colab.zip'
if goi.exists():
    print('đã có gói:', goi, goi.stat().st_size / 1e6, 'MB')
else:
    print('Chọn tệp goi_colab.zip từ máy...')
    lên = files.upload()
    ten = list(lên)[0]
    Path(ten).replace(goi)
    print('đã lưu vào', goi)

In [ ]:
import zipfile, json
GOC = Path('/content/goi')
GOC.mkdir(exist_ok=True)
with zipfile.ZipFile(goi) as z:
    z.extractall(GOC)
pham_vi = json.loads((GOC / 'pham_vi_gru.json').read_text())
print(json.dumps(pham_vi, ensure_ascii=False, indent=2))
for p in sorted(GOC.rglob('*')):
    if p.is_file():
        print(f'{p.relative_to(GOC)}  {p.stat().st_size/1e6:.2f} MB')

## 4. Kiểm dữ liệu trước khi chạy

Ba điều: mã máy, tập dòng đánh giá và thống kê chuẩn hoá. Nếu số dòng test ở đây khác số dòng của mô hình bảng thì dừng lại, đừng chạy tiếp.

In [ ]:
import pandas as pd, numpy as np
for env in pham_vi['envs']:
    p = pd.read_parquet(GOC / f'processed/{env}.parquet')
    print(f"{env}: {p['series_id'].nunique()} máy, {len(p):,} điểm, "
          f"bucket {p['bucket'].min()} … {p['bucket'].max()}")
    for h in pham_vi['horizons']:
        er = pd.read_parquet(GOC / f'eval_rows/{env}_h{h}.parquet')
        dung = er[er['bang'] & er['chuoi']]
        b0 = int(p['bucket'].min())
        te = dung[(dung['bucket'] - b0 >= 1957) & (dung['bucket'] - b0 + h < 2304)]
        print(f"   h={h}: {len(dung):,} dòng đủ cửa sổ chuỗi, {len(te):,} dòng test")

## 5. Đo nhanh tốc độ trên GPU

Chạy 20 batch của một epoch rồi suy ra thời gian. **Đừng cam kết thời gian trước khi chạy ô này.**

In [ ]:
%cd /content/goi
!python dl.py --env E2 --horizons 1 --models gru --seeds 42 \
    --goi /content/goi --out /content/do_thu \
    --epochs 1 --patience 1 --gioi-han-batch 20 --bat-buoc-gpu

In [ ]:
import json, math, pandas as pd
tt = pd.read_csv('/content/do_thu/tom_tat.csv')
ls = json.loads(tt.iloc[0]['lich_su'])
giay_20_batch = ls[0]['giay']
er = pd.read_parquet('/content/goi/eval_rows/E2_h1.parquet')
n_train = int(((er['bang'] & er['chuoi'])).sum() * 0.5)   # xấp xỉ phần train
so_batch = math.ceil(n_train / 1024)
giay_epoch = giay_20_batch / 20 * so_batch
print(f'{giay_20_batch:.1f}s cho 20 batch → ước {giay_epoch/60:.1f} phút mỗi epoch trên E2 h=1')
print(f'Bốn tổ hợp, tối đa 30 epoch, E1 lớn gấp khoảng 2,4 lần E2:')
print(f'  ước tổng tối đa {giay_epoch*30*(1+1+2.4+2.4)/3600:.1f} giờ nếu không dừng sớm')
print('Dừng sớm thường cắt còn một phần; theo dõi log ở ô sau.')

## 6. Chạy phạm vi đã chốt

Kết quả ghi thẳng vào Drive. Ô này chạy được nhiều lần: `--resume` bỏ qua tổ hợp đã xong.

In [ ]:
%cd /content/goi
!python dl.py --env E1,E2 --horizons 1,12 --models gru --seeds 42 \
    --goi /content/goi --out "/content/drive/MyDrive/cwp_gru/ket_qua" \
    --epochs 30 --patience 5 --bat-buoc-gpu --resume

## 7. Đóng gói và tải về

Tệp `ket_qua_gru.zip` chép về máy, đặt vào `results/colab/`.

In [ ]:
import zipfile
from pathlib import Path
kq = Path('/content/drive/MyDrive/cwp_gru/ket_qua')
zip_ra = Path('/content/drive/MyDrive/cwp_gru/ket_qua_gru.zip')
with zipfile.ZipFile(zip_ra, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(kq.rglob('*')):
        if p.is_file():
            z.write(p, p.relative_to(kq))
print(zip_ra, f'{zip_ra.stat().st_size/1e6:.1f} MB')
for n in zipfile.ZipFile(zip_ra).namelist():
    print(' ', n)
from google.colab import files
files.download(str(zip_ra))